# Project 2 — Phase 3: Train + Deploy the Code-Switching Language ID Model
**Code Switching NLP | Code Saviours SI-26 | Humna Imran**

*Environment: GitHub Codespaces*

Week 6 built `dataset.csv` — a labeled Roman Urdu + English code-switching dataset (`sentence`, `word`, `label`
with `label ∈ {URD, ENG, MIX}`). This notebook does the actual research part: fine-tune a token-classification
model (`xlm-roberta-base`) so it can label unseen Roman Urdu text word-by-word, publish it to the Hugging Face
Hub, and wrap it in a small Streamlit demo.

**Pipeline:**
1. Load `dataset.csv` and turn it into per-sentence word/label sequences
2. Tokenize with the XLM-RoBERTa tokenizer and align sub-word tokens back to word-level labels
3. Fine-tune `xlm-roberta-base` for token classification (3 classes: URD / ENG / MIX)
4. Evaluate — report **precision / recall / F1 per label**, which is what the submission asks for
5. Push the model + tokenizer to the Hugging Face Hub
6. Write a small Streamlit app (`app.py`) that loads the pushed model and tags typed-in sentences live
7. Deploy that app (Hugging Face Spaces or Streamlit Community Cloud) and grab the demo link

> **Note on the three files you gave me:** only `dataset.csv` is used below — it already has the final
> `sentence, word, label` columns from Week 6. `roman_urdu_dataset.csv` (the raw source corpus) and
> `google10k.txt` (the English wordlist) were inputs to the Week 6 *labeling* step, not to training, so they
> don't need to be loaded here. If you want to sanity-check that `dataset.csv` matches what Week 6 produced,
> the assertions in Step 1 below will catch it if something's off.

> **Note on GPU:** the handout says "Runtime > Change runtime type > GPU" (Colab wording). GitHub Codespaces
> doesn't have that menu — by default you get a CPU-only container. That's genuinely fine here: the Week 6
> dataset is ~220 sentences, so a few epochs of `xlm-roberta-base` finishes in single-digit minutes on CPU.
> If you want it faster, the two options are (a) pick a Codespaces machine type with a GPU if your plan/org
> offers one, or (b) run just the training cells (Steps 1–4) in a free Google Colab GPU runtime, download the
> `results/` checkpoint, and come back here for the Hub push + Streamlit steps. The code below auto-detects
> whichever device it's given, so nothing needs to change either way.


## Step 0 — Setup

In [ ]:
%pip install -q --break-system-packages transformers torch datasets scikit-learn huggingface_hub

import json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## Step 1 — Load `dataset.csv` and group into sentences

Update `DATASET_PATH` below if your Codespace layout differs — this defaults to the path you gave me.

In [ ]:
# --- config -----------------------------------------------------------
REPO_ROOT = "/workspaces/code-switching-codesaviours-si26-humna"
DATASET_PATH = f"{REPO_ROOT}/SI26-Week6/dataset.csv"
# ------------------------------------------------------------------------

df = pd.read_csv(DATASET_PATH)
assert {'sentence', 'word', 'label'}.issubset(df.columns), \
    f'Expected sentence/word/label columns, got: {list(df.columns)}'

label_list = ['URD', 'ENG', 'MIX']
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

bad_labels = set(df['label'].unique()) - set(label_list)
assert not bad_labels, f'Found unexpected labels in dataset.csv: {bad_labels}'

# Group word-by-word rows back into per-sentence word/label sequences,
# preserving original word order (no sort=True, which would scramble sentences)
grouped = df.groupby('sentence', sort=False)
sentences = [
    {'words': g['word'].astype(str).tolist(), 'labels': g['label'].tolist()}
    for _, g in grouped
]

print(f'Loaded {len(df)} word rows across {len(sentences)} sentences')
print('Label distribution (word level):')
print(df['label'].value_counts())

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)
print(f'\nTraining sentences: {len(train_data)}')
print(f'Testing sentences:  {len(test_data)}')


## Step 2 — Tokenize and align labels to sub-word tokens

XLM-RoBERTa splits words into sub-word pieces, so each word's label has to be copied onto only its *first*
sub-token (`-100` everywhere else — the Trainer's loss function ignores `-100`).

In [ ]:
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['words'], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids, prev_word = [], None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data],
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)

print(train_ds)
print(test_ds)


## Step 3 — Fine-tune XLM-RoBERTa for token classification

`compute_metrics` reports **precision / recall / F1 per label (URD, ENG, MIX)** plus a macro-F1 —
this is exactly what the submission asks you to paste into the Classroom comment.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        for p, l in zip(pred_row, label_row):
            if l != -100:
                true_labels.append(id2label[l])
                true_preds.append(id2label[p])

    report = classification_report(
        true_labels, true_preds, labels=label_list, output_dict=True, zero_division=0
    )
    metrics = {
        'accuracy': report['accuracy'],
        'f1_macro': f1_score(true_labels, true_preds, labels=label_list, average='macro', zero_division=0),
    }
    for lbl in label_list:
        metrics[f'f1_{lbl}'] = report[lbl]['f1-score']
        metrics[f'precision_{lbl}'] = report[lbl]['precision']
        metrics[f'recall_{lbl}'] = report[lbl]['recall']
    return metrics

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()
print('Training complete!')


## Step 4 — Evaluate: F1 for URD, ENG, MIX

This is the number the handout wants pasted into your Classroom submission comment.

In [ ]:
eval_results = trainer.evaluate()

print('Final evaluation metrics:')
for k, v in sorted(eval_results.items()):
    if k.startswith('eval_'):
        print(f'  {k[5:]:20s}: {v:.4f}')

with open('eval_metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print('\nSaved eval_metrics.json')
print('\nCopy this into your submission comment:')
print(f"  F1 (URD): {eval_results['eval_f1_URD']:.3f}")
print(f"  F1 (ENG): {eval_results['eval_f1_ENG']:.3f}")
print(f"  F1 (MIX): {eval_results['eval_f1_MIX']:.3f}")
print(f"  Macro F1: {eval_results['eval_f1_macro']:.3f}")


> **If MIX's F1 looks low or zero:** that's expected and worth mentioning in your submission comment,
> not a bug. `MIX` only fires on hyphenated hybrids (`type-kiya`) in the Week 6 labeler, so it's the rarest
> class by a wide margin in a 220-sentence dataset — a handful of test examples isn't enough for the model to
> learn it reliably. If you have time before Friday, the highest-leverage fix is going back to Week 6 and
> generating more MIX examples, not changing anything here.

## Step 5 — Save and push to the Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()  # paste your Hugging Face token (from https://huggingface.co/settings/tokens)


In [ ]:
HF_USERNAME = 'hamnaheh'  # same account used to publish the Week 6 dataset
MODEL_REPO_NAME = 'code-switching-langid-si26-humna'
repo_id = f'{HF_USERNAME}/{MODEL_REPO_NAME}'

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f'Model published at: https://huggingface.co/{repo_id}')


## Step 6 — Streamlit demo

Writes `app.py` and `requirements.txt` into the repo. Update `MODEL_REPO` below if you changed
`HF_USERNAME` / `MODEL_REPO_NAME` above — it needs to match what you just pushed.

In [ ]:
%%writefile app.py
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_REPO = "hamnaheh/code-switching-langid-si26-humna"  # <-- must match the repo_id you pushed to

st.set_page_config(page_title="Roman Urdu Code-Switching Language ID", page_icon="\U0001F524")

@st.cache_resource
def load_model():
    tok = AutoTokenizer.from_pretrained(MODEL_REPO)
    mdl = AutoModelForTokenClassification.from_pretrained(MODEL_REPO)
    mdl.eval()
    return tok, mdl

tokenizer, model = load_model()

COLORS = {"URD": "#2ecc71", "ENG": "#3498db", "MIX": "#e67e22"}

def predict(sentence):
    words = sentence.strip().split()
    if not words:
        return []
    encoded = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**encoded).logits
    preds = torch.argmax(logits, dim=2)[0].tolist()
    word_ids = encoded.word_ids(batch_index=0)

    results, seen = [], set()
    for idx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        results.append((words[wid], model.config.id2label[preds[idx]]))
    return results

st.title("\U0001F524 Roman Urdu Code-Switching Language ID")
st.caption("Code Saviours SI-26 \u00b7 Project 2 \u00b7 XLM-RoBERTa fine-tuned for token classification")
st.write("Type a Roman Urdu / English mixed sentence \u2014 each word gets tagged URD, ENG, or MIX.")

text = st.text_input("Sentence", "Aaj ka meeting bohot important tha yaar")

if text:
    tagged = predict(text)
    html = " ".join(
        f'<span style="background-color:{COLORS.get(l, "#bbb")};'
        f'padding:2px 6px;border-radius:4px;margin:2px;display:inline-block;">'
        f'{w} <sub>{l}</sub></span>'
        for w, l in tagged
    )
    st.markdown(html, unsafe_allow_html=True)

    st.divider()
    st.subheader("Word-by-word breakdown")
    st.table({"word": [w for w, _ in tagged], "label": [l for _, l in tagged]})


In [ ]:
%%writefile requirements.txt
streamlit
transformers
torch
huggingface_hub


### Test it locally in this Codespace (optional)

```bash
pip install --break-system-packages -r requirements.txt
streamlit run app.py --server.port 8501
```

Codespaces will prompt you to open the forwarded port in a browser tab.

## Step 7 — Deploy

Two easy options — pick whichever's faster for you, both are free:

**Option A — Hugging Face Spaces (matches the handout's "HuggingFace/streamlit" wording):**
1. Go to https://huggingface.co/new-space
2. Space name: e.g. `code-switching-langid-si26-humna-demo`
3. SDK: **Streamlit**
4. Visibility: **Public**
5. Upload `app.py` and `requirements.txt` (drag-and-drop, or `git push` if you clone the Space repo)
6. It builds automatically — your demo link will be `https://huggingface.co/spaces/<username>/<space-name>`

**Option B — Streamlit Community Cloud:**
1. Push `app.py` and `requirements.txt` to your GitHub repo (see commit command below)
2. Go to https://share.streamlit.io, sign in with GitHub, "New app"
3. Point it at this repo, branch, and `app.py` as the entry file
4. It deploys and gives you a `*.streamlit.app` link


## Step 8 — Commit and push

```bash
git add SI26-Week7-Humna.ipynb app.py requirements.txt eval_metrics.json
git commit -m "Week 7: fine-tuned XLM-RoBERTa language ID model + Streamlit demo"
git push
```


## Submission checklist

Paste these three things in Classroom by Friday:

- [ ] **Hugging Face Model Hub link** — `https://huggingface.co/hamnaheh/code-switching-langid-si26-humna` (from Step 5)
- [ ] **Week 7 notebook on GitHub** — the pushed `SI26-Week7-Humna.ipynb` (from Step 8)
- [ ] **Evaluation scores** — F1 for URD, ENG, MIX from `eval_metrics.json` / the printout in Step 4
- [ ] *(bonus, not required but nice to include)* your live demo link from Step 7
